# Fairness — formal test, Fairlearn metrics, FPDP, equivalence test

Independence/Separation/Sufficiency, standard Fairlearn metrics, FPDP diagnosis, and a
fairness equivalence (TOST) test, for whichever models have a `models/<name>_model.py` file
so far.

`test_independence`/`test_separation`/`test_sufficiency` are our own chi-square /
Fisher's-method implementations of each definition's null hypothesis, not a specific
published test statistic — treat pass/fail as directionally right, and swap in a more
specific test if the write-up needs one.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path("../..").resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.proportion import tost_proportions_2indep

import common_metrics as cm

NB_OUTPUT_DIR = cm.OUTPUT_DIR / "fairness"
NB_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    from fairlearn.metrics import (
        MetricFrame, selection_rate, true_positive_rate, false_positive_rate,
        demographic_parity_ratio, equalized_odds_difference,
    )
    HAS_FAIRLEARN = True
except ImportError:
    HAS_FAIRLEARN = False
    print("fairlearn not installed — pip install fairlearn, then re-run this cell")

cm.report_status()

## Load data

In [ ]:
train_df = cm.load_split("train")
test_df = cm.load_split("test")

X_train, y_train = cm.get_X_y(train_df)
X_test, y_test = cm.get_X_y(test_df)

print(f"train: {X_train.shape}, test: {X_test.shape}")

## 1. Formal fairness test — Independence, Separation, Sufficiency

Independence: prediction unrelated to the protected attribute (statistical parity).
Separation: prediction independent of the attribute *given the true outcome* (equal
TPR/FPR across groups). Sufficiency: true outcome independent of the attribute
*given the score* (calibration holds equally across groups).

In [ ]:
def test_independence(y_pred, group):
    table = pd.crosstab(group, y_pred)
    chi2, p, dof, expected = stats.chi2_contingency(table)
    return {"statistic": chi2, "p_value": p}


def test_separation(y_pred, y_true, group):
    p_values = []
    for outcome in [0, 1]:
        mask = y_true == outcome
        if mask.sum() == 0:
            continue
        table = pd.crosstab(group[mask], y_pred[mask])
        if table.shape[0] < 2 or table.shape[1] < 2:
            continue
        chi2, p, dof, expected = stats.chi2_contingency(table)
        p_values.append(p)
    if not p_values:
        return {"statistic": np.nan, "p_value": np.nan}
    combined_stat, combined_p = stats.combine_pvalues(p_values, method="fisher")
    return {"statistic": combined_stat, "p_value": combined_p}


def test_sufficiency(y_pred_proba, y_true, group, n_bins=10):
    try:
        bins = pd.qcut(y_pred_proba, q=n_bins, duplicates="drop")
    except ValueError:
        return {"statistic": np.nan, "p_value": np.nan}
    table = pd.crosstab([bins, group], y_true)
    try:
        chi2, p, dof, expected = stats.chi2_contingency(table)
    except ValueError:
        return {"statistic": np.nan, "p_value": np.nan}
    return {"statistic": chi2, "p_value": p}


def fairness_test(audit, group_col):
    return {
        "independence": test_independence(audit["y_pred"], audit[group_col]),
        "separation": test_separation(audit["y_pred"], audit["y_true"], audit[group_col]),
        "sufficiency": test_sufficiency(audit["y_pred_proba"], audit["y_true"], audit[group_col]),
    }

## 1b. Conditional statistical parity

Plain Independence checks whether approval rates differ by group overall, which invites the
objection "that's just different risk profiles, not discrimination." This answers it
directly: bin applicants into similar-risk bands (by DTI or LTV) first, then check whether
approval rates still differ by group *within* a band. Same chi-square-per-stratum +
Fisher's-method structure as `test_separation`, conditioning on risk band instead of outcome.

In [ ]:
RISK_SCORE_COLS = [f for f in ["debt_to_income_ratio", "combined_loan_to_value_ratio"] if f in cm.FEATURES]

def test_conditional_statistical_parity(y_pred, group, risk_score, n_bins=5):
    try:
        bins = pd.qcut(risk_score, q=n_bins, duplicates="drop")
    except ValueError:
        return {"statistic": np.nan, "p_value": np.nan}
    p_values = []
    for b in bins.cat.categories:
        mask = bins == b
        if mask.sum() == 0:
            continue
        table = pd.crosstab(group[mask], y_pred[mask])
        if table.shape[0] < 2 or table.shape[1] < 2:
            continue
        chi2, p, dof, expected = stats.chi2_contingency(table)
        p_values.append(p)
    if not p_values:
        return {"statistic": np.nan, "p_value": np.nan}
    combined_stat, combined_p = stats.combine_pvalues(p_values, method="fisher")
    return {"statistic": combined_stat, "p_value": combined_p}

## 2. Fairlearn standard metrics

Demographic parity ratio, equalized odds difference, and the disparate impact
ratio (the "4/5ths rule" — flag if below 0.8).

In [ ]:
def compute_fairlearn_metrics(audit, group_col):
    if not HAS_FAIRLEARN:
        return None
    mf = MetricFrame(
        metrics={"selection_rate": selection_rate, "tpr": true_positive_rate, "fpr": false_positive_rate},
        y_true=audit["y_true"], y_pred=audit["y_pred"], sensitive_features=audit[group_col],
    )
    dpr = demographic_parity_ratio(audit["y_true"], audit["y_pred"], sensitive_features=audit[group_col])
    eod = equalized_odds_difference(audit["y_true"], audit["y_pred"], sensitive_features=audit[group_col])
    disparate_impact_ratio = mf.by_group["selection_rate"].min() / mf.by_group["selection_rate"].max()
    return {
        "by_group": mf.by_group,
        "demographic_parity_ratio": dpr,
        "equalized_odds_difference": eod,
        "disparate_impact_ratio": disparate_impact_ratio,
    }


def plot_selection_rate(by_group, group_col, model_name):
    fig, ax = plt.subplots(figsize=(6, 3.5))
    by_group["selection_rate"].plot(kind="bar", ax=ax, color="steelblue")
    ax.set_ylabel("selection rate (approval rate)")
    ax.set_title(f"{model_name} — approval rate by {group_col}")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    fig.savefig(NB_OUTPUT_DIR / f"fairness_selection_rate_{model_name}_{group_col}.png", dpi=150, bbox_inches="tight")
    plt.show()

## 3. Fairness Partial Dependence Plot (FPDP) — candidate-variable diagnosis

Only meaningful for a (model, attribute) pair that already rejected in step 1 — this
answers *why*, not *whether*. Swept over `debt_to_income_ratio`/`combined_loan_to_value_ratio`
(the tract-level and MSA-income variables that could otherwise be prime candidates are
excluded from `X` entirely — see `common_metrics.PROTECTED_COLS`).

In [ ]:
def fairness_pdp(model, module, X, feature, group_series, group_a, group_b, n_points=20):
    disparities = []
    grid = np.linspace(X[feature].quantile(0.05), X[feature].quantile(0.95), n_points)
    for val in grid:
        X_mod = X.copy()
        X_mod[feature] = val
        preds = module.predict_proba(model, X_mod)[:, 1]
        gap = preds[(group_series == group_a).values].mean() - preds[(group_series == group_b).values].mean()
        disparities.append((val, gap))
    return disparities


def plot_fairness_pdp(disparities, feature, group_col, group_a, group_b, model_name):
    vals, gaps = zip(*disparities)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(vals, gaps, marker="o")
    ax.axhline(0, color="gray", linewidth=0.8)
    ax.set_xlabel(feature)
    ax.set_ylabel(f"P(approved | {group_a}) − P(approved | {group_b})")
    ax.set_title(f"{model_name} — FPDP — {group_col} gap vs. {feature}")
    plt.tight_layout()
    fig.savefig(NB_OUTPUT_DIR / f"fairness_fpdp_{model_name}_{group_col}_{feature}.png", dpi=150, bbox_inches="tight")
    plt.show()

## 4. Fairness Equivalence test (TOST) — primary for this dataset

With ~1.37M test rows, step 1's classical test will reject on almost any nonzero gap, so
it's not that informative here. TOST flips the null: `H0: |θ| > δ` (unfair) vs.
`H1: |θ| < δ` (fair within tolerance `δ`), via `statsmodels.tost_proportions_2indep`.

In [ ]:
def tost_two_proportions(count_a, nobs_a, count_b, nobs_b, delta):
    result = tost_proportions_2indep(count_a, nobs_a, count_b, nobs_b, low=-delta, upp=delta, compare="diff")
    return {"theta_hat": result.results_larger.diff, "tost_p": result.pvalue}


DELTA = 0.02  # team-agreed tolerance — TODO(team): confirm this value

# TODO(team): confirm these are the right comparison pairs; applicant_age and
# co_applicant_age are banded (e.g. "25-34"), so pick two specific bands before
# running the equivalence test on either.
FAIRNESS_PAIRS = {
    "derived_sex": ("Female", "Male"),
    "derived_ethnicity": ("Hispanic or Latino", "Not Hispanic or Latino"),
    "derived_race": ("Black or African American", "White"),
    "applicant_age": None,
    "co_applicant_age": None,
}

## Run the fairness pipeline across available models

logreg and TabPFN already export their test-set scores from their own training notebooks
(`get_precomputed_predictions`), so this reads those instead of re-running inference. XGBoost
has no such export, so it falls back to `get_or_fit_model` + one live `predict_proba` pass —
cheap regardless, since XGBoost isn't the slow one; avoiding TabPFN's CPU cost is the actual
point here. Every attribute's tests go through `run_step`, so one failing doesn't lose the
rest.

**Checkpointing**: same mechanism as the other two notebooks — each attribute's tests are
checkpointed under `outputs/evaluation/fairness/checkpoints/<model>/<attribute>/`, written the
moment they're computed and reused on a later run instead of recomputed. Set
`FORCE_RECOMPUTE = True` to ignore checkpoints and redo everything.

In [ ]:
def build_audit_from_precomputed(name, X_test):
    audit = cm.get_precomputed_predictions(name)
    audit["y_pred"] = (audit["y_pred_proba"] >= cm.TEAM_THRESHOLD).astype(int)
    for risk_col in RISK_SCORE_COLS:
        audit[risk_col] = X_test[risk_col].loc[audit.index]
    return audit


def build_audit_from_live(model, module, X_test, y_test, test_df):
    probs = module.predict_proba(model, X_test)[:, 1]
    audit = cm.get_audit(test_df).reset_index(drop=True)
    audit["y_true"] = y_test.reset_index(drop=True)
    audit["y_pred_proba"] = probs
    audit["y_pred"] = (probs >= cm.TEAM_THRESHOLD).astype(int)
    for risk_col in RISK_SCORE_COLS:
        audit[risk_col] = X_test[risk_col].reset_index(drop=True)
    return audit


def run_equivalence(audit, col):
    pair = FAIRNESS_PAIRS.get(col)
    if not pair:
        return None
    group_a, group_b = pair
    sub_a, sub_b = audit[audit[col] == group_a], audit[audit[col] == group_b]
    if not (len(sub_a) and len(sub_b)):
        return None
    return tost_two_proportions(sub_a["y_pred"].sum(), len(sub_a), sub_b["y_pred"].sum(), len(sub_b), DELTA)


def run_conditional_parity(audit, col):
    return {
        risk_col: test_conditional_statistical_parity(audit["y_pred"], audit[col], audit[risk_col])
        for risk_col in RISK_SCORE_COLS
    }


FORCE_RECOMPUTE = False  # set True to ignore checkpoints and recompute everything

fairness_results = {}

for name, module in cm.available_models().items():
    print(f"\n=== {name} ===")
    model = None
    precomputed = cm.get_precomputed_predictions(name)
    if precomputed is not None:
        print(f"  using precomputed predictions from disk ({len(precomputed)} rows) — no live inference")
        audit = build_audit_from_precomputed(name, X_test)
    else:
        try:
            model = cm.get_or_fit_model(name, module, X_train, y_train)
        except Exception as e:
            print(f"  model unavailable ({type(e).__name__}: {e}) — skipping {name} entirely")
            continue
        audit = build_audit_from_live(model, module, X_test, y_test, test_df)

    approval_rate = audit["y_pred"].mean()
    print(f"  approval rate at cm.TEAM_THRESHOLD={cm.TEAM_THRESHOLD:.3f}: {approval_rate:.4f}")

    model_results = {"approval_rate": approval_rate}
    for col in cm.AUDIT_COLS:
        entry = {}
        ckpt_dir = NB_OUTPUT_DIR / "checkpoints" / name / col
        cm.run_step(entry, "test", fairness_test, audit, col, checkpoint_dir=ckpt_dir, force=FORCE_RECOMPUTE)
        cm.run_step(entry, "conditional_parity", run_conditional_parity, audit, col, checkpoint_dir=ckpt_dir, force=FORCE_RECOMPUTE)
        if HAS_FAIRLEARN:
            cm.run_step(entry, "fairlearn", compute_fairlearn_metrics, audit, col, checkpoint_dir=ckpt_dir, force=FORCE_RECOMPUTE)
        cm.run_step(entry, "equivalence", run_equivalence, audit, col, checkpoint_dir=ckpt_dir, force=FORCE_RECOMPUTE)

        if "test" in entry:
            p_indep = entry["test"]["independence"]["p_value"]
            flag = " ← REJECTS fairness (p < 0.05)" if p_indep < 0.05 else ""
            print(f"  {col}: independence p={p_indep:.4g}{flag}")
        if "fairlearn" in entry and entry["fairlearn"] is not None:
            plot_selection_rate(entry["fairlearn"]["by_group"], col, name)
        if "conditional_parity" in entry:
            for risk_col, result in entry["conditional_parity"].items():
                cp_flag = " ← still rejects, conditioning on risk doesn't explain it" if result["p_value"] < 0.05 else ""
                print(f"    conditional on {risk_col}: p={result['p_value']:.4g}{cp_flag}")

        # FPDP only makes sense once something's rejected, so fetch a live model lazily
        # here instead of unconditionally — and only once per model, reused across attributes.
        rejects = "test" in entry and entry["test"]["independence"]["p_value"] < 0.05
        pair = FAIRNESS_PAIRS.get(col)
        if rejects and pair:
            if model is None:
                try:
                    model = cm.get_or_fit_model(name, module, X_train, y_train)
                except Exception as e:
                    print(f"    FPDP skipped — model unavailable ({type(e).__name__}: {e})")
            if model is not None:
                entry["fpdp"] = {}
                fpdp_sample = X_test.sample(min(3000, len(X_test)), random_state=42)
                group_series = test_df.loc[fpdp_sample.index, col]
                group_a, group_b = pair
                for feature in RISK_SCORE_COLS:
                    if cm.run_step(entry["fpdp"], feature, fairness_pdp, model, module, fpdp_sample, feature, group_series, group_a, group_b, checkpoint_dir=ckpt_dir / "fpdp", force=FORCE_RECOMPUTE):
                        plot_fairness_pdp(entry["fpdp"][feature], feature, col, group_a, group_b, name)

        model_results[col] = entry

    fairness_results[name] = model_results

if not fairness_results:
    print("No models ready yet — drop a models/<name>_model.py file in and re-run.")

## Cross-model comparison — disparate impact ratio, with the 4/5ths line marked

In [ ]:
if fairness_results and HAS_FAIRLEARN:
    rows = []
    for name, model_results in fairness_results.items():
        for col in cm.AUDIT_COLS:
            entry = model_results.get(col, {})
            if "fairlearn" in entry and entry["fairlearn"] is not None:
                rows.append({"model": name, "attribute": col, "disparate_impact_ratio": entry["fairlearn"]["disparate_impact_ratio"]})
    if rows:
        dir_df = pd.DataFrame(rows)
        pivot = dir_df.pivot(index="attribute", columns="model", values="disparate_impact_ratio")
        fig, ax = plt.subplots(figsize=(8, 5))
        pivot.plot(kind="bar", ax=ax)
        ax.axhline(0.8, color="red", ls="--", lw=1, label="4/5ths threshold")
        ax.set_ylabel("disparate impact ratio (min/max selection rate)")
        ax.set_title("Disparate impact ratio by model and protected attribute")
        ax.legend()
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        fig.savefig(NB_OUTPUT_DIR / "fairness_disparate_impact_ratio_comparison.png", dpi=150, bbox_inches="tight")
        plt.show()

## Summary table — one row per model, copy-ready for the slide deck

`approval rate` is worth watching — the same `TEAM_THRESHOLD` can imply very different
approval rates per model depending on how spread out each model's probabilities are (see
`common_metrics.py`).

In [ ]:
summary_rows = []
for name, model_results in fairness_results.items():
    worst_dir, worst_p = np.nan, np.nan
    for col in cm.AUDIT_COLS:
        entry = model_results.get(col, {})
        if "fairlearn" in entry and entry["fairlearn"] is not None:
            dir_val = entry["fairlearn"]["disparate_impact_ratio"]
            worst_dir = dir_val if np.isnan(worst_dir) else min(worst_dir, dir_val)
        if "test" in entry:
            p = entry["test"]["independence"]["p_value"]
            worst_p = p if np.isnan(worst_p) else min(worst_p, p)
    summary_rows.append({
        "model": name,
        "approval rate": round(model_results.get("approval_rate", float("nan")), 4),
        "worst independence p-value": round(worst_p, 4) if not np.isnan(worst_p) else "—",
        "worst disparate impact ratio": round(worst_dir, 3) if not np.isnan(worst_dir) else "—",
    })

fairness_summary_df = pd.DataFrame(summary_rows)
fairness_summary_df

## Smoke test — remove once real models are in `models/`

Runs the same pipeline on a throwaway logistic regression and a small sample,
purely to check the harness works end to end.

In [ ]:
if not fairness_results:
    print("Running a throwaway smoke test — NOT a real model, just checking the harness works end to end.")
    sample = train_df.sample(20_000, random_state=42)
    Xs, ys = cm.get_X_y(sample)
    test_sample = test_df.sample(5_000, random_state=42)
    Xt, yt = cm.get_X_y(test_sample)
    smoke_model = cm.SmokeTestModule.fit(Xs, ys)
    smoke_probs = cm.SmokeTestModule.predict_proba(smoke_model, Xt)[:, 1]
    smoke_audit = cm.get_audit(test_sample).reset_index(drop=True)
    smoke_audit["y_true"] = yt.reset_index(drop=True)
    smoke_audit["y_pred_proba"] = smoke_probs
    smoke_audit["y_pred"] = (smoke_probs >= cm.TEAM_THRESHOLD).astype(int)
    print(fairness_test(smoke_audit, "derived_sex"))
    print("Harness is wired correctly.")

## Save outputs

Writes to `outputs/evaluation/fairness/` (gitignored, same as `models/` and `data/`) so
results survive independently of the notebook. Individual attribute results are already
checkpointed as they run (see above); this writes the final consolidated summary + full
results for convenience. No model objects are stored in `fairness_results`, so the whole
thing is pickled as-is.

In [ ]:
import pickle

if fairness_results:
    fairness_summary_df.to_csv(NB_OUTPUT_DIR / "fairness_summary.csv", index=False)
    with open(NB_OUTPUT_DIR / "fairness_results.pkl", "wb") as f:
        pickle.dump(fairness_results, f)
    print(f"Saved summary + full results to {NB_OUTPUT_DIR}/")